In [8]:
from fractions import Fraction
from typing import List, Tuple, Optional

def to_frac(x):
    """Преобразует число (int/float/str/Fraction) в Fraction."""
    return x if isinstance(x, Fraction) else Fraction(x)

def simplex_canonical(
    c: List,                  # коэффициенты ЦФ (максимизация), длина = n
    A: List[List],            # матрица ограничений (m x n), равенства
    b: List                   # правые части (длина = m), b >= 0
) -> Optional[Tuple[List[Fraction], Fraction]]:
    """
    Решает задачу ЛП в канонической форме:
        максимизировать c^T x
        при условиях A x = b, x >= 0,
        где начальный базисный допустимый план задан (например, единичная подматрица в A).
    Предполагается, что первые m базисных переменных соответствуют единичной подматрице
    (или пользователь гарантирует корректность начального базиса).
    """
    # Преобразуем всё во Fraction
    c = [to_frac(x) for x in c]
    A = [[to_frac(x) for x in row] for row in A]
    b = [to_frac(x) for x in b]

    m = len(A)      # число уравнений
    n = len(c)      # число переменных

    assert len(b) == m, "Длина b должна совпадать с числом строк A"
    assert all(len(row) == n for row in A), "Несогласованность размеров A и c"
    assert all(bi >= 0 for bi in b), "Требуется b >= 0"

    # Определяем начальный базис: ищем единичную подматрицу
    # Для простоты предположим, что базис — последние m переменных
    # (это стандартное соглашение при ручном приведении к каноническому виду)
    basis = list(range(n - m, n))

    # Проверка: действительно ли столбцы базиса образуют единичную матрицу?
    # (необязательно, но полезно для отладки)
    for i in range(m):
        for j in range(m):
            expected = Fraction(1) if i == j else Fraction(0)
            actual = A[i][basis[j]]
            if actual != expected:
                print(f"Предупреждение: столбец базисной переменной x{basis[j]} "
                      f"в строке {i} равен {actual}, ожидалось {expected}. "
                      "Продолжаем, но корректность базиса — на вас.")

    iteration = 0
    while True:
        iteration += 1
        print(f"\n--- Итерация {iteration} ---")

        # Строим симплекс-таблицу: (m+1) × (n+1)
        table = [[Fraction(0)] * (n + 1) for _ in range(m + 1)]

        # Строки ограничений
        for i in range(m):
            for j in range(n):
                table[i + 1][j] = A[i][j]
            table[i + 1][n] = b[i]

        # Начальное заполнение строки ЦФ: z - c^T x = 0 → коэффициенты = -c_j
        for j in range(n):
            table[0][j] = -c[j]

        # Корректируем строку 0: добавляем Σ c_Bi * (строка i)
        current_z = Fraction(0)
        for i in range(m):
            cb = c[basis[i]]
            current_z += cb * b[i]
            for j in range(n + 1):
                table[0][j] += cb * table[i + 1][j]

        # Вывод таблицы с именами строк
        header = "\t" + "\t".join([f"x{j}" for j in range(n)] + ["RHS"])
        print(header)
        print(f"z\t" + "\t".join(str(val) for val in table[0]))
        for i in range(m):
            basic_var = basis[i]
            print(f"x{basic_var}\t" + "\t".join(str(val) for val in table[i + 1]))
        print(f"Текущее значение целевой функции: {current_z}")

        # Проверка оптимальности: все коэффициенты в строке 0 >= 0?
        if all(table[0][j] >= 0 for j in range(n)):
            print("Оптимальное решение найдено.")
            solution = [Fraction(0)] * n
            for i in range(m):
                solution[basis[i]] = b[i]
            return solution, current_z

        # Выбор ведущего столбца (входит в базис) — первый отрицательный
        entering = next(j for j in range(n) if table[0][j] < 0)
        print(f"Входит переменная x{entering}")

        # Проверка на неограниченность
        if all(table[i + 1][entering] <= 0 for i in range(m)):
            print("Целевая функция не ограничена сверху.")
            return None

        # Выбор ведущей строки (покидает базис) — правило минимального отношения
        ratios = [
            (table[i + 1][n] / table[i + 1][entering], i)
            for i in range(m)
            if table[i + 1][entering] > 0
        ]
        _, leaving_row = min(ratios, key=lambda x: x[0])
        leaving_var = basis[leaving_row]
        print(f"Покидает базис переменная x{leaving_var}")

        # Обновляем базис
        basis[leaving_row] = entering

        # Приводим ведущую строку к единице
        pivot = table[leaving_row + 1][entering]
        for j in range(n + 1):
            table[leaving_row + 1][j] /= pivot

        # Обнуляем ведущий столбец в остальных строках
        for i in range(m + 1):
            if i == leaving_row + 1:
                continue
            factor = table[i][entering]
            for j in range(n + 1):
                table[i][j] -= factor * table[leaving_row + 1][j]

        # Обновляем A и b для следующей итерации
        for i in range(m):
            for j in range(n):
                A[i][j] = table[i + 1][j]
            b[i] = table[i + 1][n]

In [9]:
if __name__ == "__main__":
    c = [1, 3, 1, 0, 0, 0]
    A = [
        [5, 3, 0, 1, 0, 0],
        [1, 2, 4, 0, 1, 0],
        [0, 1, 1, 0, 0, 1]
    ]
    b = [8, 4, 1]

    sol, val = simplex_canonical(c, A, b)
    print("\nРешение:", [str(x) for x in sol])
    print("Оптимальное значение:", val)


--- Итерация 1 ---
	x0	x1	x2	x3	x4	x5	RHS
z	-1	-3	-1	0	0	0	0
x3	5	3	0	1	0	0	8
x4	1	2	4	0	1	0	4
x5	0	1	1	0	0	1	1
Текущее значение целевой функции: 0
Входит переменная x0
Покидает базис переменная x3

--- Итерация 2 ---
	x0	x1	x2	x3	x4	x5	RHS
z	0	-12/5	-1	1/5	0	0	8/5
x0	1	3/5	0	1/5	0	0	8/5
x4	0	7/5	4	-1/5	1	0	12/5
x5	0	1	1	0	0	1	1
Текущее значение целевой функции: 8/5
Входит переменная x1
Покидает базис переменная x5

--- Итерация 3 ---
	x0	x1	x2	x3	x4	x5	RHS
z	0	0	7/5	1/5	0	12/5	4
x0	1	0	-3/5	1/5	0	-3/5	1
x4	0	0	13/5	-1/5	1	-7/5	1
x1	0	1	1	0	0	1	1
Текущее значение целевой функции: 4
Оптимальное решение найдено.

Решение: ['1', '1', '0', '0', '1', '0']
Оптимальное значение: 4
